In [1]:
import pandas as pd
import glob
import os
import numpy as np

print("--- Starting Master Panel Data Construction (2007 - 2024) ---")

# 1. ปรับปรุงรายการคอลัมน์แกนหลัก (เปลี่ยน hhid -> qid และเพิ่ม qid ในรายชื่อระบุตัวตน)
core_panel_columns = [
    'interview__key', 'qid', 'survey_year', 'shocks_Group',
    'impact_3way', 'impact_2way', 'recovery_std', 'cons_label',
    'coping_savings', 'coping_insurance', 'coping_informal_borrow', 
    'coping_formal_borrow', 'coping_assets', 'coping_gov_help'
]

all_wave_data = []

# 2. ค้นหาไฟล์ CSV ที่คลีนเสร็จแล้วทั้งหมดในเครื่อง
csv_files = glob.glob('shocks_*_cleaned.csv')

for file_path in sorted(csv_files):
    print(f"Processing and extracting core panel variables from: {file_path}")
    df_wave = pd.read_csv(file_path)
    
    # ปรับชื่อคอลัมน์ Coping และชื่อเรียกตัวแปรให้ตรงกัน
    rename_dict = {
        'coping_used_savings': 'coping_savings',
        'coping_used_insurance': 'coping_insurance',
        'coping_borrowed_informal': 'coping_informal_borrow',
        'coping_borrowed_formal': 'coping_formal_borrow',
        'coping_sold_assets': 'coping_assets',
        'consumption_label': 'cons_label',
        'hhid': 'qid'  # Fallback: ถ้าปีไหนใช้ชื่อ hhid ให้แปลงสภาพเป็น qid รอไว้ก่อน
    }
    df_wave = df_wave.rename(columns=rename_dict)
    
    # ตรรกะกรณีไม่พบคอลัมน์ qid หรือ hhid เลย ให้ดึง interview__key มาเป็นรหัสครัวเรือนแทน
    if 'qid' not in df_wave.columns and 'interview__key' in df_wave.columns:
        df_wave['qid'] = df_wave['interview__key']
        
    # คัดเฉพาะคอลัมน์ที่มีอยู่ในข้อมูลจริงและอยู่ในกลุ่มแกนหลัก
    existing_cols = [col for col in core_panel_columns if col in df_wave.columns]
    df_filtered = df_wave[existing_cols].copy()
    
    all_wave_data.append(df_filtered)

# 3. รวมร่างข้อมูลในแนวตั้ง (Longitudinal Form)
master_panel_df = pd.concat(all_wave_data, axis=0, ignore_index=True)

# ตรวจสอบและสร้างคอลัมน์ที่ขาดหายให้เป็น NaN เพื่อป้องกันโครงสร้างตารางพัง
for col in core_panel_columns:
    if col not in master_panel_df.columns:
        master_panel_df[col] = np.nan

# จัดเรียงลำดับคอลัมน์มาตรฐานก่อนเริ่มกระบวนการจัดรูป Wide Format
master_panel_df = master_panel_df[core_panel_columns]

print(f"\n[Long Format] Total Longitudinal Observations: {len(master_panel_df)} shock events.")

# -------------------------------------------------------------------------
# 4. 🔥 โค้ดส่วนใหม่: แปลงข้อมูลจาก Long เป็น Wide (1 Row per qid-year)
# -------------------------------------------------------------------------
print("\n--- Reshaping Panel Data to Wide Format (Collapsing Multiple Shocks) ---")

# กำหนดคีย์ระบุตัวตนประจำแผงข้อมูลครัวเรือนรายปี
id_keys = ['qid', 'survey_year']

# ระบุกลุ่มคอลัมน์ข้อมูลภัยพิบัติที่ต้องการจะแตกลำดับคอลัมน์เพิ่ม (e.g., _1, _2)
reshape_target_cols = [
    'shocks_Group', 'impact_3way', 'impact_2way', 'recovery_std', 'cons_label',
    'coping_savings', 'coping_insurance', 'coping_informal_borrow', 
    'coping_formal_borrow', 'coping_assets', 'coping_gov_help'
]

# คลีนค่าว่างในคีย์หลักออกไปก่อนเพื่อไม่ให้เกิดข้อผิดพลาดในการ Groupby
master_panel_df = master_panel_df.dropna(subset=id_keys)

# สร้างตัวนับลำดับเหตุการณ์ภัยพิบัติ (Shock Sequence) แยกรายครัวเรือนรายปี
# หากครัวเรือนเจอรอบเดียวจะได้เลข 1, หากเจอซ้ำจะรันเลข 2, 3, ... ไปเรื่อยๆ 
master_panel_df['shock_seq'] = master_panel_df.groupby(id_keys).cumcount() + 1

# ดำเนินการ Pivot ข้อมูลด้วยความเร็วสูง (Vectorized Pivot)
master_panel_wide = master_panel_df.pivot(
    index=id_keys,
    columns='shock_seq',
    values=reshape_target_cols
)

# ยุบคอลัมน์ MultiIndex ให้เป็นชื่อชั้นเดียวแบบอ่านง่าย เช่น shocks_Group_1, impact_3way_1
master_panel_wide.columns = [f"{col}_{seq}" for col, seq in master_panel_wide.columns]

# ดึงดัชนี qid และ survey_year กลับขึ้นมาเป็นคอลัมน์ปกติ
master_panel_wide = master_panel_wide.reset_index()

# ตรวจสอบการพังทลายของข้อมูลหลังจากการบีบอัดแถว
print(f"✨ Master Wide Panel Construction Complete! ✨")
print(f"Total Lean Household-Year Rows: {len(master_panel_wide)}")
print(f"Data Breakdown by Survey Year (Unique Households Count):")
print(master_panel_wide['survey_year'].value_counts().sort_index())

# 5. ส่งออกไฟล์แผงข้อมูลสุดท้ายในรูปแบบ Wide Format
output_wide_file = 'master_panel_shocks_2007_2024_wide.csv'
master_panel_wide.to_csv(output_wide_file, index=False)
print(f"\n💾 Saved wide panel file successfully as: {output_wide_file}")

--- Starting Master Panel Data Construction (2007 - 2024) ---
Processing and extracting core panel variables from: shocks_2007_cleaned.csv
Processing and extracting core panel variables from: shocks_2008_cleaned.csv
Processing and extracting core panel variables from: shocks_2010_cleaned.csv
Processing and extracting core panel variables from: shocks_2011_cleaned.csv
Processing and extracting core panel variables from: shocks_2013_cleaned.csv
Processing and extracting core panel variables from: shocks_2016_cleaned.csv
Processing and extracting core panel variables from: shocks_2017_cleaned.csv
Processing and extracting core panel variables from: shocks_2019_cleaned.csv
Processing and extracting core panel variables from: shocks_2022_cleaned.csv
Processing and extracting core panel variables from: shocks_2024_cleaned.csv

[Long Format] Total Longitudinal Observations: 23057 shock events.

--- Reshaping Panel Data to Wide Format (Collapsing Multiple Shocks) ---
✨ Master Wide Panel Constr

/tmp/ipykernel_8943/1175720104.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_wave['qid'] = df_wave['interview__key']


In [1]:
import pandas as pd

# 1. โหลดไฟล์ผลลัพธ์ที่ได้ออกมาตรวจสอบ
df_result = pd.read_csv('master_panel_shocks_2007_2024_wide.csv')

print("====== 📊 1. ตรวจสอบโครงสร้างพื้นฐาน ======")
print(f"จำนวนแถวทั้งหมด (Household-Year): {df_result.shape[0]:,}")
print(f"จำนวนคอลัมน์ทั้งหมด: {df_result.shape[1]:,}")

print("\n====== 🆔 2. พิสูจน์รหัสตัวตน (Q ID แทน HH ID) ======")
# เช็คว่ามีคอลัมน์ qid และ survey_year อยู่จริงไหม และมี hhid หลงเหลืออยู่หรือเปล่า
has_qid = 'qid' in df_result.columns
has_year = 'survey_year' in df_result.columns
has_hhid = 'hhid' in df_result.columns

print(f"-> พบคอลัมน์ 'qid' หรือไม่: {has_qid} (ต้องเป็น True)")
print(f"-> พบคอลัมน์ 'survey_year' หรือไม่: {has_year} (ต้องเป็น True)")
print(f"-> คอลัมน์ 'hhid' ดั้งเดิมถูกกำจัดไปแล้วใช่ไหม: {not has_hhid} (ต้องเป็น True)")

print("\n====== 🔄 3. ตรวจสอบการยุบแถวซ้ำ (1 Row per Household-Year) ======")
# ตรวจสอบว่าใน 1 ปี มี qid ไหนโผล่ซ้ำมากกว่า 1 แถวหรือไม่
duplicate_check = df_result.duplicated(subset=['qid', 'survey_year']).sum()
print(f"-> จำนวนแถวที่ซ้ำซ้อนของ [qid + survey_year]: {duplicate_check}")
if duplicate_check == 0:
    print("✅ ยอดเยี่ยม! ทุกครัวเรือนมีเพียง 1 แถวต่อ 1 ปีอย่างเด็ดขาดตามโจทย์")
else:
    print("⚠️ เตือน: ยังมีข้อมูลซ้ำซ้อนอยู่")

print("\n====== 🔀 4. ตรวจสอบการกระจายคอลัมน์ (Wide Format) ======")
# ดึงรายชื่อคอลัมน์ที่มีคำว่า 'shocks_Group' ออกมาดูว่าแตกตัวออกไปสูงสุดกี่ลำดับ
shock_cols = [col for col in df_result.columns if 'shocks_Group' in col]
print(f"-> คอลัมน์ลำดับของ Shock ที่เกิดขึ้นจริงใน Dataset: {shock_cols}")
print(f"-> ครัวเรือนในแผงข้อมูลนี้ เคยเจอภัยพิบัติสูงสุดพร้อมกันใน 1 ปี = {len(shock_cols)} ครั้ง")

print("\n====== 👀 5. สุ่มตัวอย่างข้อมูล 3 แถวแรก ======")
# คัดเฉพาะคอลัมน์สำคัญมาพรีวิวดูความสวยงาม
preview_cols = ['qid', 'survey_year'] + [c for c in shock_cols[:2]]
print(df_result[preview_cols].head(3))

====== 📊 1. ตรวจสอบโครงสร้างพื้นฐาน ======
จำนวนแถวทั้งหมด (Household-Year): 10,777
จำนวนคอลัมน์ทั้งหมด: 145

====== 🆔 2. พิสูจน์รหัสตัวตน (Q ID แทน HH ID) ======
-> พบคอลัมน์ 'qid' หรือไม่: True (ต้องเป็น True)
-> พบคอลัมน์ 'survey_year' หรือไม่: True (ต้องเป็น True)
-> คอลัมน์ 'hhid' ดั้งเดิมถูกกำจัดไปแล้วใช่ไหม: True (ต้องเป็น True)

====== 🔄 3. ตรวจสอบการยุบแถวซ้ำ (1 Row per Household-Year) ======
-> จำนวนแถวที่ซ้ำซ้อนของ [qid + survey_year]: 0
✅ ยอดเยี่ยม! ทุกครัวเรือนมีเพียง 1 แถวต่อ 1 ปีอย่างเด็ดขาดตามโจทย์

====== 🔀 4. ตรวจสอบการกระจายคอลัมน์ (Wide Format) ======
-> คอลัมน์ลำดับของ Shock ที่เกิดขึ้นจริงใน Dataset: ['shocks_Group_1', 'shocks_Group_2', 'shocks_Group_3', 'shocks_Group_4', 'shocks_Group_5', 'shocks_Group_6', 'shocks_Group_7', 'shocks_Group_8', 'shocks_Group_9', 'shocks_Group_10', 'shocks_Group_11', 'shocks_Group_12', 'shocks_Group_13']
-> ครัวเรือนในแผงข้อมูลนี้ เคยเจอภัยพิบัติสูงสุดพร้อมกันใน 1 ปี = 13 ครั้ง

====== 👀 5. สุ่มตัวอย่างข้อมูล 3 แถวแรก ======
    qid 

/tmp/ipykernel_9761/2682698550.py:4: DtypeWarning: Columns (0: qid, 1: shocks_Group_10, 2: shocks_Group_11, 3: shocks_Group_12, 4: shocks_Group_13, 5: recovery_std_1, 6: recovery_std_2, 7: recovery_std_3, 8: recovery_std_4, 9: recovery_std_5, 10: recovery_std_6, 11: recovery_std_7, 12: recovery_std_8, 13: recovery_std_9, 14: cons_label_1, 15: cons_label_2, 16: cons_label_3, 17: cons_label_4, 18: cons_label_5, 19: cons_label_6, 20: cons_label_7, 21: cons_label_8, 22: cons_label_9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_result = pd.read_csv('master_panel_shocks_2007_2024_wide.csv')


Previous code to create panal all line is at this point until end.

In [1]:
import pandas as pd
import glob
import os

print("--- Starting Master Panel Data Construction (2007 - 2024) ---")

# 1. กำหนดรายชื่อคอลัมน์มาตรฐานที่เราต้องการเก็บไว้ใน Panel Data 
# เพื่อคัดกรองตัวแปรขยะของแต่ละระลอกออกไป ไม่ให้ไฟล์บวม
core_panel_columns = [
    'interview__key', 'hhid', 'survey_year', 'shocks_Group',
    'impact_3way', 'impact_2way', 'recovery_std', 'cons_label',
    'coping_savings', 'coping_insurance', 'coping_informal_borrow', 
    'coping_formal_borrow', 'coping_assets', 'coping_gov_help'
]

all_wave_data = []

# 2. ค้นหาไฟล์ CSV ที่คลีนเสร็จแล้วทั้งหมดในเครื่อง
# หมายเหตุ: มั่นใจว่าชื่อไฟล์คลีนของคุณระบุปีตรงตามตรรกะนี้ เช่น shocks_2007_cleaned.csv
csv_files = glob.glob('shocks_*_cleaned.csv')

for file_path in sorted(csv_files):
    print(f"Processing and extracting core panel variables from: {file_path}")
    df_wave = pd.read_csv(file_path)
    
    # ตรวจเช็คกรณีชื่อคอลัมน์ Coping ในยุคเก่าที่อาจไม่ได้ถูกปรับระหว่างทาง ให้สอดคล้องกันก่อน Append
    rename_dict = {
        'coping_used_savings': 'coping_savings',
        'coping_used_insurance': 'coping_insurance',
        'coping_borrowed_informal': 'coping_informal_borrow',
        'coping_borrowed_formal': 'coping_formal_borrow',
        'coping_sold_assets': 'coping_assets',
        'consumption_label': 'cons_label'
    }
    df_wave = df_wave.rename(columns=rename_dict)
    
    # หากระลอกใดไม่มีคอลัมน์ระบุ hhid ดั้งเดิม ให้พยายามเก็บ ID พื้นฐานไว้สำหรับการระบุครัวเรือน
    if 'hhid' not in df_wave.columns and 'interview__key' in df_wave.columns:
        df_wave['hhid'] = df_wave['interview__key']
        
    # คัดเฉพาะคอลัมน์ที่มีอยู่ในข้อมูลจริงและอยู่ในกลุ่มแกนหลัก
    existing_cols = [col for col in core_panel_columns if col in df_wave.columns]
    df_filtered = df_wave[existing_cols]
    
    all_wave_data.append(df_filtered)

# 3. รวมร่างข้อมูลในแนวตั้งข้ามช่วงเวลา (Longitudinal Form)
master_panel_df = pd.concat(all_wave_data, axis=0, ignore_index=True)

# เติมคอลัมน์ที่ขาดหายในบาง Wave ให้เป็น NaN เพื่อรักษาโครงสร้างตาราง
for col in core_panel_columns:
    if col not in master_panel_df.columns:
        master_panel_df[col] = np.nan

# จัดเรียงคอลัมน์ให้สวยงามและเป็นระเบียบ
master_panel_df = master_panel_df[core_panel_columns]

print(f"\n✨ Master Panel Construction Complete! ✨")
print(f"Total Longitudinal Observations: {len(master_panel_df)} shock events across time.")
print(f"Data Breakdown by Survey Year:")
print(master_panel_df['survey_year'].value_counts().sort_index())

# 4. ส่งออกไฟล์แผงข้อมูลที่สมบูรณ์ที่สุด
output_master_file = 'master_panel_shocks_2007_2024.csv'
master_panel_df.to_csv(output_master_file, index=False)
print(f"\n💾 Saved master file successfully as: {output_master_file}")

--- Starting Master Panel Data Construction (2007 - 2024) ---
Processing and extracting core panel variables from: shocks_2007_cleaned.csv
Processing and extracting core panel variables from: shocks_2008_cleaned.csv
Processing and extracting core panel variables from: shocks_2010_cleaned.csv
Processing and extracting core panel variables from: shocks_2011_cleaned.csv
Processing and extracting core panel variables from: shocks_2013_cleaned.csv
Processing and extracting core panel variables from: shocks_2016_cleaned.csv
Processing and extracting core panel variables from: shocks_2017_cleaned.csv
Processing and extracting core panel variables from: shocks_2019_cleaned.csv
Processing and extracting core panel variables from: shocks_2022_cleaned.csv
Processing and extracting core panel variables from: shocks_2024_cleaned.csv

✨ Master Panel Construction Complete! ✨
Total Longitudinal Observations: 23057 shock events across time.
Data Breakdown by Survey Year:
survey_year
2007.0    2020
2008

/var/folders/7l/zffgp8nd1wbb6qkk7lg__0_c0000gn/T/ipykernel_9704/3162969437.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_wave['hhid'] = df_wave['interview__key']
